In [ ]:
# =============================================================================
# CRISP-DM - FASE 3: PREPARACIÓN DE LOS DATOS
# Proyecto: Análisis de la Depresión en Estudiantes y sus Predictores
# =============================================================================

# -----------------------------------------------------------------------------
# PREPARAR EL ENTORNO
# -----------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from scipy.stats import norm
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Carga de datos (Ajusta la ruta según tu ubicación de archivo)
ruta_csv = 'Depresión en Estudiantes.csv'
df = pd.read_csv(ruta_csv)

# -----------------------------------------------------------------------------
# CONCLUSIONES DE LA FASE 2: COMPRENSIÓN DE LOS DATOS
# -----------------------------------------------------------------------------
"""
Hallazgos clave que justifican esta fase:
1. La columna 'Id' es irrelevante para el poder predictivo.
2. 'Estrés Financiero' es decimal (float64) pero debe ser categórico entero (int64).
3. Existen nulos en 'Ciudad' y 'Estrés Financiero'.
4. 'Duración del Sueño' contiene valores atípicos erróneos (99.00 y 3.00).
5. 'Pensamientos Suicidas?' tiene formatos inconsistentes (mezcla de 0/1 con No/Si).
"""

# -----------------------------------------------------------------------------
# 1. ELIMINACIÓN DE COLUMNAS INNECESARIAS
# -----------------------------------------------------------------------------
# Eliminamos 'Id' ya que es un identificador administrativo, no un predictor.
df.drop(columns=['Id'], inplace=True)

# -----------------------------------------------------------------------------
# 2. MANEJO DE DATOS FALTANTES (NULOS)
# -----------------------------------------------------------------------------
# A. Asignar Moda a 'Ciudad' (Variable Categórica).
moda_ciudad = df['Ciudad'].mode()
df['Ciudad'].fillna(moda_ciudad, inplace=True)

# B. Eliminar filas con nulos en 'Estrés Financiero' por ser pocos registros.
df.dropna(subset=['Estrés Financiero'], inplace=True)

# -----------------------------------------------------------------------------
# 3. MANEJO DE VALORES ERRÓNEOS Y REEMPLAZO
# -----------------------------------------------------------------------------
# A. Corregir 'Duración del Sueño': Eliminar valores imposibles como 99.00.
df = df[~df['Duración del Sueño'].isin([99.00, 3.00])]

# B. Estandarizar 'Pensamientos Suicidas?': Unificar formatos 0/1 a No/Si.
df['Pensamientos Suicidas?'] = df['Pensamientos Suicidas?'].replace({'0': 'No', '1': 'Si'})

# C. Ajustar rango de 'Estrés Financiero': Iniciar en 0 para consistencia.
# Restamos 1 para que el rango [1-5] pase a ser [0-4].
df['Estrés Financiero'] = df['Estrés Financiero'] - 1

# -----------------------------------------------------------------------------
# 4. CAMBIO DE TIPOS DE VARIABLES
# -----------------------------------------------------------------------------
# Convertir 'Estrés Financiero' a entero para cálculos categóricos.
df['Estrés Financiero'] = df['Estrés Financiero'].astype(int)

# -----------------------------------------------------------------------------
# 5. CODIFICACIÓN DE VARIABLES CATEGÓRICAS (TRANSFORMACIÓN)
# -----------------------------------------------------------------------------
# Codificación Ordinal para mantener jerarquía donde aplique.

def codificar_ordinal(columna, orden):
    serie_cat = pd.Categorical(df[columna], categories=orden, ordered=True)
    return serie_cat.codes

# Codificando variables clave
df['GEN'] = codificar_ordinal('Género', ['Masculino', 'Femenino'])
df['DS'] = codificar_ordinal('Duración del Sueño', ['Menos de 5 horas', '5-6 horas', '7-8 horas', 'Más de 8 horas'])
df['PS'] = codificar_ordinal('Pensamientos Suicidas?', ['No', 'Si'])
df['AFEM'] = codificar_ordinal('Antecedentes Familiares de Enfermedades Mentales', ['No', 'Si'])

# -----------------------------------------------------------------------------
# 6. INGENIERÍA DE CARACTERÍSTICAS (FEATURE ENGINEERING)
# -----------------------------------------------------------------------------
# Crear una variable derivada que combine Presión Académica y Satisfacción.
# Hipótesis: Baja satisfacción + Alta presión = Mayor vulnerabilidad.
df['Carga_Mental'] = df['Presión Académica'] / (df['Satisfacción con los Estudios'] + 1)

# -----------------------------------------------------------------------------
# 7. ESCALADO DE VARIABLES (NORMALIZACIÓN)
# -----------------------------------------------------------------------------
# Escalamos 'Promedio de Calificaciones Acumulativo' al rango [1].
scaler = MinMaxScaler()
df['Promedio_Normalizado'] = scaler.fit_transform(df[['Promedio de Calificaciones Acumulativo']])

# -----------------------------------------------------------------------------
# VERIFICACIÓN FINAL Y GUARDADO
# -----------------------------------------------------------------------------
print("Resumen de nulos finales:\n", df.isnull().sum())
print("\nDimensiones finales del dataset:", df.shape)

# Guardar el dataset preparado
df.to_csv('Dataset_Preparado_Depresion.csv', index=False)
print("\nArchivo 'Dataset_Preparado_Depresion.csv' guardado con éxito.")